In [6]:
import ants
import torch
import torchio as tio
import random
import numpy as np
import torchio.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
array = np.zeros((9, 9, 9))
#array[3:6, 3:6, 3] = 1
array[5, 0, 5] = 1
corners = [
    (0, 0, 0), (0, 0, 8), (0, 8, 0), (0, 8, 8),
    (8, 0, 0), (8, 0, 8), (8, 8, 0), (8, 8, 8)
]
for corner in corners:
    array[corner] = 1

array=torch.tensor(array).unsqueeze(0)
print(array)

array2 = np.random.rand(9, 9, 9)
array2=torch.tensor(array2).unsqueeze(0)

# Set seeds for reproducibility
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
np.random.seed(0)
random.seed(0)

tensor([[[[1., 0., 0., 0., 0., 0., 0., 0., 1.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [1., 0., 0., 0., 0., 0., 0., 0., 1.]],

         [[0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0

In [7]:
path_mask = '/home/user/Documents/raph/preprocessed_datasets/ISLES2022/derivatives/sub-121/ses-0001/sub-121_ses-0001_msk.nii.gz'
path_img = '/home/user/Documents/raph/preprocessed_datasets/ISLES2022/sub-121/ses-0001/anat/sub-121_ses-0001_FLAIR.nii.gz'

mask = ants.image_read(path_mask)
img = ants.image_read(path_img)

mask = mask.numpy()
img = img.numpy()

mask = torch.tensor(mask).unsqueeze(0)
img = torch.tensor(img).unsqueeze(0)

In [8]:
new_data=tio.LabelMap(tensor=array)

subject_dict = {
    'image': tio.ScalarImage(tensor=img),
    'mask': tio.LabelMap(tensor=mask),
}

subject = tio.Subject(subject_dict)

dataset = tio.SubjectsDataset([subject])

label_sampler = tio.data.LabelSampler(patch_size=128,
                                    label_name='mask', 
                                    label_probabilities={0: 3, 1: 4})

patches_training_set=tio.Queue(
    dataset,
    max_length=50,
    samples_per_volume=10,
    sampler=label_sampler,
    num_workers=1,
    shuffle_subjects=True,
    shuffle_patches=True,
)

In [14]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Assuming patches_training_set is defined
training_loader = DataLoader(patches_training_set, batch_size=4, shuffle=True)

for batch in training_loader:
    masks = batch['mask']['data']  # Assuming the key for the images is 'mask'
    masks_np = masks.numpy()  # Convert to NumPy array for easier slicing and visualization
    
    # Determine the number of patches in the batch
    num_patches = masks_np.shape[0]
    
    def update_plot(slice_index):
        fig, axs = plt.subplots(1, num_patches, figsize=(15, num_patches * 3))
        for i in range(num_patches):
            slice_2d = masks_np[i, 0, slice_index, :, :]  # Use the slider's value to select the slice
            axs[i].imshow(slice_2d, cmap='gray')
            axs[i].axis('off')
        plt.show()

    # Assuming the patches are 3D, create a slider to select the slice
    slice_slider = widgets.IntSlider(min=0, max=masks_np.shape[2]-1, step=1, value=masks_np.shape[2] // 2, description='Slice')
    
    # Display the widget and link it to the update function
    interactive_plot = widgets.interactive(update_plot, slice_index=slice_slider)
    display(interactive_plot)

    # Break after the first batch to avoid creating widgets for all batches
    #break

interactive(children=(IntSlider(value=64, description='Slice', max=127), Output()), _dom_classes=('widget-inte…

interactive(children=(IntSlider(value=64, description='Slice', max=127), Output()), _dom_classes=('widget-inte…

interactive(children=(IntSlider(value=64, description='Slice', max=127), Output()), _dom_classes=('widget-inte…

New LABELSAMPLER


In [17]:
label_sampler = tio.data.sampler.LabelSampler_Pad4LabelPatches(
    patch_size=128,
    label_name='mask',
    label_probabilities={0: 3, 1: 4},

)

patches_training_set=tio.Queue(
    dataset,
    max_length=50,
    samples_per_volume=10,
    sampler=label_sampler,
    num_workers=1,
    shuffle_subjects=True,
    shuffle_patches=True,
)

In [18]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Assuming patches_training_set is defined
training_loader = DataLoader(patches_training_set, batch_size=4, shuffle=True)

def update_plot(slice_index):
    fig, axs = plt.subplots(1, num_patches, figsize=(15, num_patches * 3))
    for i in range(num_patches):
        slice_2d = masks_np[i, 0, slice_index, :, :]  # Use the slider's value to select the slice
        axs[i].imshow(slice_2d, cmap='gray')
        axs[i].axis('off')
    plt.show()

for batch in training_loader:
    masks = batch['mask']['data']  # Assuming the key for the images is 'mask'
    masks_np = masks.numpy()  # Convert to NumPy array for easier slicing and visualization
    
    if np.any(masks_np > 0):
        print("3D image with at least one value > 0:")
        print(masks_np)

    # Determine the number of patches in the batch
    num_patches = masks_np.shape[0]

    # Assuming the patches are 3D, create a slider to select the slice
    slice_slider = widgets.IntSlider(min=0, max=masks_np.shape[2]-1, step=1, value=masks_np.shape[2] // 2, description='Slice')
    
    # Display the widget and link it to the update function
    interactive_plot = widgets.interactive(update_plot, slice_index=slice_slider)
    display(interactive_plot)

    # Break after the first batch to avoid creating widgets for all batches
    break

3D image with at least one value > 0:
[[[[[0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    ...
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]]

   [[0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    ...
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]]

   [[0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    ...
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]]

   ...

   [[0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    ...
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]]

   [[0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    ...
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]]

   [[0. 0. 0. ... 0. 0. 0.]
    [0. 0. 0. ... 0. 0. 0.]
    [0. 

interactive(children=(IntSlider(value=64, description='Slice', max=127), Output()), _dom_classes=('widget-inte…